[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/19_3d_pointcloud_and_detection.ipynb)

# 19. 3D point-cloud and detection ops — local geometry matters

이전 Point Transformer section은 모든 point feature에 그냥 global scaled-dot-product attention을 적용했기 때문에 Point Transformer의 핵심인 **local neighborhood + relative 3D position encoding**이 빠져 있었다. PointNet++도 FPS만 있고 grouping/local aggregation이 없었다.

이번 버전은 PointNet → PointNet++ set abstraction → DGCNN EdgeConv → Point Transformer local attention → voxel/pillar BEV → CenterPoint-style 3D box decode 순서로 연결한다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. PointNet: shared point MLP + symmetric pooling


In [ ]:
points = torch.tensor(
    [
        [0.0, 0.0, 0.0],
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.0, 0.0, 1.0],
        [1.0, 1.0, 0.2],
        [0.8, 0.2, 0.7],
    ],
    device=device,
)

point_mlp = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
).to(device)

point_features = point_mlp(points)
global_feature = point_features.max(dim=0).values

print("point features:", point_features.shape)
print("global feature:", global_feature.shape)


## 2. PointNet++ set abstraction: FPS → neighborhood grouping → local PointNet

PointNet++는 global max pooling만 하는 PointNet에 local hierarchy를 추가한다. 먼저 FPS로 centroid를 고르고, centroid 주변 point를 grouping한 뒤 상대 좌표를 shared MLP + max pool로 local feature로 만든다.


In [ ]:
def farthest_point_sampling(points, num_centroids):
    selected = [0]
    min_distance = torch.cdist(
        points,
        points[[0]],
    ).squeeze(1)

    for _ in range(num_centroids - 1):
        next_index = int(min_distance.argmax())
        selected.append(next_index)

        distance_to_new = torch.cdist(
            points,
            points[[next_index]],
        ).squeeze(1)
        min_distance = torch.minimum(
            min_distance,
            distance_to_new,
        )

    return torch.tensor(selected, device=points.device)


centroid_ids = farthest_point_sampling(points, num_centroids=2)
centroids = points[centroid_ids]

distance = torch.cdist(centroids, points)
neighbor_ids = distance.topk(
    k=3,
    largest=False,
).indices
neighbors = points[neighbor_ids]
relative_xyz = neighbors - centroids[:, None, :]

local_mlp = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 24),
).to(device)

local_features = local_mlp(relative_xyz)
centroid_features = local_features.max(dim=1).values

print("centroid ids:", centroid_ids)
print("neighbor ids:\n", neighbor_ids)
print("PointNet++ local features:", centroid_features.shape)


## 3. DGCNN EdgeConv

EdgeConv는 center feature와 neighbor-center difference를 함께 MLP에 넣고 neighbor dimension에서 max aggregation한다. graph를 feature space에서 다시 구성하면 dynamic graph가 된다.


In [ ]:
pairwise_distance = torch.cdist(points, points)
knn_ids = pairwise_distance.topk(
    k=3,
    largest=False,
).indices[:, 1:]

center = points[:, None, :].expand(-1, knn_ids.size(1), -1)
neighbor = points[knn_ids]
edge_input = torch.cat(
    [center, neighbor - center],
    dim=-1,
)

edge_mlp = nn.Sequential(
    nn.Linear(6, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
).to(device)

edge_features = edge_mlp(edge_input).max(dim=1).values
print("EdgeConv output:", edge_features.shape)


## 4. Point Transformer: local vector attention with relative position

Point Transformer는 point set에 vanilla global attention을 적용하는 것이 아니다. query point `i`의 local neighbors `j`에 대해 relative coordinate `p_i-p_j`를 positional MLP로 변환하고, attention weight와 value 양쪽에 geometry를 넣는다.


In [ ]:
feature_dim = 16
features = point_features

query_projection = nn.Linear(feature_dim, feature_dim).to(device)
key_projection = nn.Linear(feature_dim, feature_dim).to(device)
value_projection = nn.Linear(feature_dim, feature_dim).to(device)

position_mlp = nn.Sequential(
    nn.Linear(3, feature_dim),
    nn.ReLU(),
    nn.Linear(feature_dim, feature_dim),
).to(device)
attention_mlp = nn.Sequential(
    nn.Linear(feature_dim, feature_dim),
    nn.ReLU(),
    nn.Linear(feature_dim, feature_dim),
).to(device)

q = query_projection(features)
k = key_projection(features)
v = value_projection(features)

neighbor_k = k[knn_ids]
neighbor_v = v[knn_ids]
neighbor_xyz = points[knn_ids]

relative_position = points[:, None, :] - neighbor_xyz
position_encoding = position_mlp(relative_position)

attention_input = (
    q[:, None, :]
    - neighbor_k
    + position_encoding
)
attention_logits = attention_mlp(attention_input)
attention_weights = attention_logits.softmax(dim=1)

point_transformer_output = (
    attention_weights
    * (neighbor_v + position_encoding)
).sum(dim=1)

print("local neighbors per point:", knn_ids.size(1))
print("vector attention weights:", attention_weights.shape)
print("Point Transformer output:", point_transformer_output.shape)


## 5. Voxelization and pillar/BEV aggregation

PointPillars 계열은 continuous xyz를 grid cell/pillar index로 양자화하고 같은 cell의 point features를 aggregate한 뒤 2D BEV pseudo-image에 scatter한다.


In [ ]:
voxel_size = 0.5
voxel_index = torch.floor(points / voxel_size).to(torch.int64)

xy_index = voxel_index[:, :2]
xy_index = xy_index - xy_index.min(dim=0).values

grid_height = int(xy_index[:, 1].max()) + 1
grid_width = int(xy_index[:, 0].max()) + 1
bev = torch.zeros(
    feature_dim,
    grid_height,
    grid_width,
    device=device,
)

for point_index in range(points.size(0)):
    x_index = int(xy_index[point_index, 0])
    y_index = int(xy_index[point_index, 1])
    bev[:, y_index, x_index] += features[point_index]

print("voxel indices:\n", voxel_index)
print("BEV feature map:", bev.shape)


## 6. CenterPoint-style 3D box decode

CenterPoint는 BEV center heatmap peak만 찾는 것으로 끝나지 않는다. offset으로 continuous xy center를 복원하고, height/z, 3D dimensions, rotation head를 함께 읽어 oriented 3D box parameter를 만든다.


In [ ]:
heatmap = torch.tensor(
    [[0.1, 0.7], [0.2, 0.9]],
    device=device,
)
offset_x = torch.tensor(
    [[0.1, 0.2], [0.0, -0.1]],
    device=device,
)
offset_y = torch.tensor(
    [[0.0, 0.1], [0.2, 0.3]],
    device=device,
)
height_z = torch.tensor(
    [[0.5, 0.6], [0.4, 0.8]],
    device=device,
)
dimensions = torch.tensor(
    [2.0, 4.2, 1.6],
    device=device,
)
rotation_sin = torch.tensor(0.6, device=device)
rotation_cos = torch.tensor(0.8, device=device)

score, flat_index = heatmap.flatten().topk(1)
width = heatmap.size(1)
grid_y = flat_index // width
grid_x = flat_index % width

y_index = int(grid_y.item())
x_index = int(grid_x.item())

center_x = grid_x.float() + offset_x[y_index, x_index]
center_y = grid_y.float() + offset_y[y_index, x_index]
center_z = height_z[y_index, x_index]
yaw = torch.atan2(rotation_sin, rotation_cos)

box = torch.cat(
    [
        center_x.reshape(1),
        center_y.reshape(1),
        center_z.reshape(1),
        dimensions,
        yaw.reshape(1),
    ]
)

print("score:", score.item())
print("[x, y, z, w, l, h, yaw]:", box)


## References and provenance

**PointNet / PointNet++** — Qi et al. shared point MLP, symmetric pooling, FPS, local grouping, local PointNet aggregation을 반영했다.

**DGCNN** — Wang et al. center/neighbor-difference EdgeConv와 neighborhood aggregation을 반영했다.

**Point Transformer** — Zhao et al. local neighborhood, relative xyz positional encoding, vector attention, geometry-added values를 반영했다.

**PointPillars / CenterPoint** — pillar/BEV aggregation과 center heatmap + offset + z + dimensions + yaw decoding의 핵심을 반영했다. production detector는 voxel feature encoder, sparse/dense backbone, NMS 등의 추가 구조를 가진다.
